<a href="https://colab.research.google.com/github/JamesSembukuttiarachchi/rain_in_australia/blob/main/01_logistic_regression_weatherAUS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **1. Import Libraries**

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

### **2. Load Dataset**

In [ ]:

# Dataset should be uploaded to /content in Colab
df = pd.read_csv('/content/weatherAUS_balanced.csv')

# Display basic info
print("Dataset Shape:", df.shape)
df.head()

### **3. Basic Cleaning**

In [ ]:

# Drop rows where target is missing (if any)
df = df.dropna(subset=['RainTomorrow'])

# Convert target variable to binary
df['RainTomorrow'] = df['RainTomorrow'].map({'No': 0, 'Yes': 1})

# Optional: encode RainToday if exists
if 'RainToday' in df.columns:
    df['RainToday'] = df['RainToday'].map({'No': 0, 'Yes': 1})

### **4. Feature Engineering**

In [ ]:

# Convert Date column to useful features
if 'Date' in df.columns:
    df['Date'] = pd.to_datetime(df['Date'])
    df['Year'] = df['Date'].dt.year
    df['Month'] = df['Date'].dt.month
    df['Day'] = df['Date'].dt.day
    df = df.drop(columns=['Date'])

### **5. Split Features & Target**

In [ ]:

X = df.drop('RainTomorrow', axis=1)
y = df['RainTomorrow']

### **6. Identify Column Types**

In [ ]:

numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object']).columns

### **7. Preprocessing Pipeline**

In [ ]:

# Logistic Regression performs better with scaled features
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

### **8. Train-Test Split**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

### **9. Model (Optimized)**

In [ ]:

# Using 'liblinear' for stability + better convergence
model = LogisticRegression(
    max_iter=500,
    solver='liblinear'
)

# Combine preprocessing + model
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', model)
])

### **10. Train Model**

In [ ]:

pipeline.fit(X_train, y_train)

### **11. Predictions**

In [ ]:

y_pred = pipeline.predict(X_test)

### **12. Evaluation**

In [ ]:

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))